<a href="https://colab.research.google.com/github/exaucee-m/Big-Data-CW/blob/main/Coursework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Mounting google drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [1]:
#Session set up
import pyspark
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StringIndexer, StandardScaler
from pyspark.sql.functions import *

#Spark session
spark = SparkSession.builder \
    .appName("consumer_complaints") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.memory.offHeap.enabled", "true") \
    .config("spark.memory.offHeap.size", "2g") \
    .getOrCreate()


In [2]:
#Ingestion & parquet
df = spark.read \
.option("header", True) \
.option("multiLine", True) \
.option("quote",'"') \
.option("escape", '"') \
.csv('/content/drive/MyDrive/complaints.csv', inferSchema = True)

df.write.mode("overwrite").parquet("/content/drive/MyDrive/complaints.parquet")
df = spark.read.parquet("/content/drive/MyDrive/complaints.parquet")


In [3]:
#Checking schema - columns and types etc
df.show(5)
df.printSchema()

+-------------+--------------------+--------------------+--------------------+--------------------+----------------------------+-----------------------+--------------------+-----+--------+-------------+--------------------------+-------------+--------------------+----------------------------+----------------+------------------+------------+
|Date received|             Product|         Sub-product|               Issue|           Sub-issue|Consumer complaint narrative|Company public response|             Company|State|ZIP code|         Tags|Consumer consent provided?|Submitted via|Date sent to company|Company response to consumer|Timely response?|Consumer disputed?|Complaint ID|
+-------------+--------------------+--------------------+--------------------+--------------------+----------------------------+-----------------------+--------------------+-----+--------+-------------+--------------------------+-------------+--------------------+----------------------------+----------------+----

Features of choice:
Product, Sub-product, Issue, Sub-issue, State, Submitted via, Company response to consumer.

In [4]:
#Checking dataset - rows
df.cache()
df.count()

13689875

In [5]:
# This shows the values in column (yes, no and n/a)
df.select("Consumer disputed?").distinct().show(20, truncate=False)


+------------------+
|Consumer disputed?|
+------------------+
|No                |
|N/A               |
|Yes               |
+------------------+



In [6]:
#Clean data - consumer disputed shows only yes or no
df = df.filter(col("Consumer disputed?").isin("Yes", "No"))
df.show(20)

+-------------+--------------------+--------------------+--------------------+--------------------+----------------------------+-----------------------+--------------------+-----+--------+--------------------+--------------------------+-------------+--------------------+----------------------------+----------------+------------------+------------+
|Date received|             Product|         Sub-product|               Issue|           Sub-issue|Consumer complaint narrative|Company public response|             Company|State|ZIP code|                Tags|Consumer consent provided?|Submitted via|Date sent to company|Company response to consumer|Timely response?|Consumer disputed?|Complaint ID|
+-------------+--------------------+--------------------+--------------------+--------------------+----------------------------+-----------------------+--------------------+-----+--------+--------------------+--------------------------+-------------+--------------------+----------------------------+

In [7]:
#EDA - groupBy - checking how many yes vs nos after removing n/a values
df.groupBy ("Consumer disputed?").count().show()

+------------------+------+
|Consumer disputed?| count|
+------------------+------+
|                No|619817|
|               Yes|148378|
+------------------+------+



In [8]:
#EDA - Check for missing values
import pyspark.sql.functions as F
df.select([F.count(F.when(F.col(c).isNull(),c)).alias(c) for c in df.columns]).show()

+-------------+-------+-----------+-----+---------+----------------------------+-----------------------+-------+-----+--------+------+--------------------------+-------------+--------------------+----------------------------+----------------+------------------+------------+
|Date received|Product|Sub-product|Issue|Sub-issue|Consumer complaint narrative|Company public response|Company|State|ZIP code|  Tags|Consumer consent provided?|Submitted via|Date sent to company|Company response to consumer|Timely response?|Consumer disputed?|Complaint ID|
+-------------+-------+-----------+-----+---------+----------------------------+-----------------------+-------+-----+--------+------+--------------------------+-------------+--------------------+----------------------------+----------------+------------------+------------+
|            0|      0|     235155|    0|   455341|                      604217|                 572486|      0| 5650|    3857|659731|                        38|            0|

In [9]:
#Clean data - drop all the null values
df = df.dropna()
df.show(20)

+-------------+---------------+--------------------+--------------------+--------------------+----------------------------+-----------------------+--------------------+-----+--------+--------------------+--------------------------+-------------+--------------------+----------------------------+----------------+------------------+------------+
|Date received|        Product|         Sub-product|               Issue|           Sub-issue|Consumer complaint narrative|Company public response|             Company|State|ZIP code|                Tags|Consumer consent provided?|Submitted via|Date sent to company|Company response to consumer|Timely response?|Consumer disputed?|Complaint ID|
+-------------+---------------+--------------------+--------------------+--------------------+----------------------------+-----------------------+--------------------+-----+--------+--------------------+--------------------------+-------------+--------------------+----------------------------+---------------

In [10]:
#Repartitions
df = df.repartition(10)

In [11]:
#Feature engineering and scaling

from pyspark.ml.feature import StringIndexer, VectorAssembler

for col_name in df.columns:
    if col_name.endswith("_index"):
        df = df.drop(col_name)

if "label" in df.columns:
    df = df.drop("label")

if "features" in df.columns:
    df = df.drop("features")

features = [
    "Product",
    "Sub-product",
    "Issue",
    "Sub-issue",
    "State",
    "Submitted via",
    "Company response to consumer"
]

for col_name in features:
    indexer = StringIndexer(
        inputCol=col_name,
        outputCol=col_name + "_index",
        handleInvalid="keep"
    )
    df = indexer.fit(df).transform(df)


assembler = VectorAssembler(
    inputCols=[col_name + "_index" for col_name in features],
    outputCol="features"
)

df = assembler.transform(df)

label_indexer = StringIndexer(
    inputCol="Consumer disputed?",
    outputCol="label"
)

df = label_indexer.fit(df).transform(df)

df.select("label", "features").show(5)
df.printSchema()

+-----+--------------------+
|label|            features|
+-----+--------------------+
|  0.0|       (7,[4],[6.0])|
|  0.0|(7,[2,3,4],[1.0,5...|
|  1.0|(7,[2,3,4],[5.0,7...|
|  1.0|[0.0,4.0,3.0,4.0,...|
|  0.0|      (7,[4],[22.0])|
+-----+--------------------+
only showing top 5 rows
root
 |-- Date received: date (nullable = true)
 |-- Product: string (nullable = true)
 |-- Sub-product: string (nullable = true)
 |-- Issue: string (nullable = true)
 |-- Sub-issue: string (nullable = true)
 |-- Consumer complaint narrative: string (nullable = true)
 |-- Company public response: string (nullable = true)
 |-- Company: string (nullable = true)
 |-- State: string (nullable = true)
 |-- ZIP code: string (nullable = true)
 |-- Tags: string (nullable = true)
 |-- Consumer consent provided?: string (nullable = true)
 |-- Submitted via: string (nullable = true)
 |-- Date sent to company: date (nullable = true)
 |-- Company response to consumer: string (nullable = true)
 |-- Timely response?: stri

In [12]:
#Train/Test Split - 80/20 prevents overfitting,
# seed ensures reproducibility
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

In [13]:
# Classification model - Logistic Regression
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(
    featuresCol="features",
    labelCol="label"
)
lr_model = lr.fit(train_df)
lr_preds = lr_model.transform(test_df)

In [14]:
#Classification model - Random Forest Classifier
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=200,
    maxDepth=10,
    maxBins =128
)
rf_model = rf.fit(train_df)
rf_preds = rf_model.transform(test_df)

In [15]:
#Classification model - Gradient-Boosted Tree Classifier
from pyspark.ml.classification import GBTClassifier

gbt = GBTClassifier(
    featuresCol="features",
    labelCol="label",
    maxIter=20,
    maxDepth=5,
    maxBins=128
)
gbt_model = gbt.fit(train_df)
gbt_preds = gbt_model.transform(test_df)

In [16]:
#Classification model - Linear SVM
from pyspark.ml.classification import LinearSVC

svm = LinearSVC(
    featuresCol="features",
    labelCol="label"
)
svm_model = svm.fit(train_df)
svm_preds = svm_model.transform(test_df)

In [17]:
#Evaluation - accuracy
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

print("Logistic Regression Accuracy:", evaluator.evaluate(lr_preds))
print("Random Forest Accuracy:", evaluator.evaluate(rf_preds))
print("Gradient-Boosted Tree Accuracy:", evaluator.evaluate(gbt_preds))
print("Linear SVM Accuracy:", evaluator.evaluate(svm_preds))


Logistic Regression Accuracy: 0.7996715927750411
Random Forest Accuracy: 0.8013136288998358
Gradient-Boosted Tree Accuracy: 0.7832512315270936
Linear SVM Accuracy: 0.7996715927750411


In [18]:
#Evaluation - F1 scores (precision vs recall)
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator_f1 = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

print("LR F1 Score:", evaluator_f1.evaluate(lr_preds))
print("RF F1 Score:", evaluator_f1.evaluate(rf_preds))
print("GBT F1 Score:", evaluator_f1.evaluate(gbt_preds))
print("Linear SVM F1 Score:", evaluator_f1.evaluate(svm_preds))

LR F1 Score: 0.7106570541632208
RF F1 Score: 0.7175548714990303
GBT F1 Score: 0.729082558613282
Linear SVM F1 Score: 0.7106570541632208
